# Adaptive A\* for Mobile-Robot Global Path Planning — Step-by-Step Re-Implementation

**Paper reproduced:** H. Cao, Z. Guo, Y. Zhang, Z. Lu, and L. Jiang, "An Adaptive A\* Algorithm for Mobile Robots Global Path Planning," *Electronics*, vol. 15, no. 13, Art. 2807, 2026, doi: 10.3390/electronics15132807.

The paper has no public source code, so this notebook re-implements its method from scratch and reproduces its experiments. It runs top to bottom with **Run All**, on CPU only, with no Internet and no uploaded files.

| Part | What it does |
|---|---|
| 1 | Setup |
| 2 | Implementation: grid model, heuristics + **adaptive weight (Eq. 6)**, **hybrid distance (Eq. 7)**, A\* / Dijkstra / bidirectional A\*, **redundant point deletion (Sec. 3.3)**, metrics, planners |
| 3 | Sanity checks (correctness tests) |
| 4 | Maps: proxy maps calibrated to the paper's statistics |
| 5 | The paper's published results (Tables 1–5) |
| 6 | Experiments: Table 1 ablation, Tables 2–4 algorithm comparison, Table 5 start/goal points |
| 7 | Headline comparison: published vs. reproduced (Metrics 1–4) |
| 8 | Paired timing study |
| 9 | Robustness on 150 random maps (Wilcoxon tests) |
| 10 | Sensitivity to the ambiguous pruning rule |
| 11 | Summary |

## 1. Setup

In [ ]:
# Run configuration. QUICK = True gives a ~1-minute smoke run; use False for the reported numbers.
QUICK = False
TIMING_REPEATS = 3 if QUICK else 30     # timing repetitions per configuration (median reported)
TIMING_PAIRS = 30 if QUICK else 300     # interleaved A* / Adaptive A* pairs in the timing study
ROBUST_MAPS = 5 if QUICK else 50        # random maps per scene type in the robustness study

In [ ]:
%matplotlib inline
import gc, heapq, io, itertools, math, platform, statistics, time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import scipy
from scipy.stats import wilcoxon
from IPython.display import display, Markdown

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__}")
print("CPU:", platform.processor() or platform.machine(), "| cores:", __import__("os").cpu_count())

## 2. Implementation

### 2.1 Environment model (paper Sec. 2.1)
The workspace is a grid of 1 m × 1 m cells; black cells are obstacles. The robot moves in **8 directions** (straight cost 1, diagonal cost √2).

**Assumption (not stated in the paper):** a diagonal move may not cut between two obstacles that touch at a corner.

In [ ]:
"""Occupancy-grid environment model (paper Sec. 2.1).

Cells are 1 m x 1 m squares. Internally a map is a boolean NumPy array
``occ[y, x]`` (True = obstacle) with 0-based indices. The paper uses 1-based
``(x, y)`` coordinates with the origin at the bottom-left, e.g. start ``(2, 2)``
on every map and goal ``(99, 99)`` on the 100 x 100 map; ``from_paper`` and
``to_paper`` convert between the two conventions.
"""


import math
from pathlib import Path

import numpy as np

SQRT2 = math.sqrt(2.0)

# 8-connected motion model (assumption M2): straight moves cost 1, diagonal sqrt(2).
STRAIGHT_MOVES = ((1, 0), (-1, 0), (0, 1), (0, -1))
DIAGONAL_MOVES = ((1, 1), (1, -1), (-1, 1), (-1, -1))

Cell = tuple[int, int]


def from_paper(x: int, y: int) -> Cell:
    """Convert a 1-based paper coordinate to a 0-based internal cell."""
    return (x - 1, y - 1)


def to_paper(cell: Cell) -> tuple[int, int]:
    """Convert a 0-based internal cell to a 1-based paper coordinate."""
    return (cell[0] + 1, cell[1] + 1)


class GridMap:
    """A 2-D occupancy grid with an 8-connected (or 4-connected) neighbourhood.

    Parameters
    ----------
    occ:
        Boolean array indexed ``[y, x]``; True marks an obstacle cell.
    connectivity:
        8 (default, as implied by the paper's 45-degree turn metric) or 4.
    allow_corner_cutting:
        If False (default, assumption M3) a diagonal move is only allowed when
        both orthogonally adjacent cells are free, so the robot never squeezes
        between two obstacles that touch at a corner.
    """

    def __init__(self, occ: np.ndarray, connectivity: int = 8,
                 allow_corner_cutting: bool = False, name: str = "map"):
        if connectivity not in (4, 8):
            raise ValueError("connectivity must be 4 or 8")
        self.occ = np.asarray(occ, dtype=bool)
        self.height, self.width = self.occ.shape
        self.connectivity = connectivity
        self.allow_corner_cutting = allow_corner_cutting
        self.name = name
        # Plain nested lists are much faster than NumPy for scalar look-ups
        # inside the Python search loop.
        self._occ_rows = self.occ.tolist()

    # ------------------------------------------------------------------ cells
    def in_bounds(self, x: int, y: int) -> bool:
        return 0 <= x < self.width and 0 <= y < self.height

    def is_free(self, x: int, y: int) -> bool:
        return 0 <= x < self.width and 0 <= y < self.height and not self._occ_rows[y][x]

    def neighbors(self, cell: Cell):
        """Yield ``(neighbour_cell, step_cost)`` pairs for a free cell."""
        x, y = cell
        rows = self._occ_rows
        w, h = self.width, self.height
        for dx, dy in STRAIGHT_MOVES:
            nx, ny = x + dx, y + dy
            if 0 <= nx < w and 0 <= ny < h and not rows[ny][nx]:
                yield (nx, ny), 1.0
        if self.connectivity == 4:
            return
        for dx, dy in DIAGONAL_MOVES:
            nx, ny = x + dx, y + dy
            if not (0 <= nx < w and 0 <= ny < h) or rows[ny][nx]:
                continue
            if not self.allow_corner_cutting and (rows[y][nx] or rows[ny][x]):
                continue
            yield (nx, ny), SQRT2

    # -------------------------------------------------------------------- I/O
    def save_csv(self, path: str | Path) -> None:
        """Save as CSV with the top row = largest y (so the file looks like the figure)."""
        np.savetxt(path, self.occ[::-1].astype(int), fmt="%d", delimiter=",")

    @classmethod
    def load_csv(cls, path: str | Path, **kwargs) -> "GridMap":
        arr = np.loadtxt(path, delimiter=",", dtype=int)[::-1]
        kwargs.setdefault("name", Path(path).stem)
        return cls(arr.astype(bool), **kwargs)

    def obstacle_ratio(self) -> float:
        return float(self.occ.mean())

    def __repr__(self) -> str:
        return (f"GridMap(name={self.name!r}, size={self.width}x{self.height}, "
                f"obstacles={self.obstacle_ratio():.1%}, connectivity={self.connectivity})")

### 2.2 Distance models, adaptive weight and hybrid heuristic (Sec. 2.2, 3.1, 3.2)

**Contribution 1 — adaptive estimation function (Eqs. 5–6):**
$$f(n) = g(n) + \omega(n)\,h(n), \qquad \omega(n) = \ln\!\left(1 + e^{d/D}\right)$$
where *d* is the distance from node *n* to the goal and *D* is the distance from start to goal.

**Contribution 2 — hybrid distance model (Eq. 7):**
$$h(n) = \frac{\text{Euclidean}(n, goal) + \text{Manhattan}(n, goal)}{2}$$

In [ ]:
"""Distance models and the adaptive heuristic weight (paper Sec. 2.2, 3.1, 3.2)."""


import math
from typing import Callable


DistanceFn = Callable[[Cell, Cell], float]
WeightFn = Callable[[Cell], float]


def euclidean(a: Cell, b: Cell) -> float:
    """Eq. (2): straight-line distance."""
    return math.hypot(a[0] - b[0], a[1] - b[1])


def manhattan(a: Cell, b: Cell) -> float:
    """Eq. (3): city-block distance."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def chebyshev(a: Cell, b: Cell) -> float:
    """Eq. (4): maximum axis difference."""
    return max(abs(a[0] - b[0]), abs(a[1] - b[1]))


def octile(a: Cell, b: Cell) -> float:
    """Exact 8-connected cost on an empty grid (not used by the paper; reference only)."""
    dx, dy = abs(a[0] - b[0]), abs(a[1] - b[1])
    return max(dx, dy) + (math.sqrt(2.0) - 1.0) * min(dx, dy)


def hybrid(a: Cell, b: Cell) -> float:
    """Eq. (7): arithmetic mean of the Euclidean and Manhattan distances."""
    dx, dy = abs(a[0] - b[0]), abs(a[1] - b[1])
    return 0.5 * (math.hypot(dx, dy) + dx + dy)


def zero(a: Cell, b: Cell) -> float:
    """h = 0: A* degenerates into Dijkstra's algorithm."""
    return 0.0


DISTANCES: dict[str, DistanceFn] = {
    "euclidean": euclidean,
    "manhattan": manhattan,
    "chebyshev": chebyshev,
    "octile": octile,
    "hybrid": hybrid,
    "zero": zero,
}


def adaptive_weight(start: Cell, goal: Cell, distance: DistanceFn = euclidean) -> WeightFn:
    """Eq. (6): omega(n) = ln(1 + exp(d / D)).

    ``d`` is the distance from node ``n`` to the goal and ``D`` the distance from
    the start to the goal, both measured with ``distance`` (assumption M4:
    Euclidean). omega = ln(1 + e) ~ 1.313 at the start and ln 2 ~ 0.693 at the goal.
    """
    big_d = distance(start, goal)
    if big_d == 0:
        return lambda n: 1.0

    def omega(n: Cell) -> float:
        return math.log1p(math.exp(distance(n, goal) / big_d))

    return omega


def constant_weight(value: float = 1.0) -> WeightFn:
    return lambda n: value

In [ ]:
# What does the adaptive weight look like?
w = adaptive_weight((0, 0), (10, 0))
print(f"omega at the start (d/D = 1): {w((0, 0)):.4f}   = ln(1+e)")
print(f"omega half way    (d/D = .5): {w((5, 0)):.4f}")
print(f"omega at the goal (d/D = 0): {w((10, 0)):.4f}   = ln 2")
# How much does the hybrid heuristic over-estimate the true 8-connected cost?
ratios = [hybrid((0, 0), (math.cos(t), math.sin(t))) / octile((0, 0), (math.cos(t), math.sin(t)))
          for t in np.linspace(0, math.pi / 2, 181)]
print(f"max hybrid / true cost = {max(ratios):.4f}  (at 45 degrees) -> the hybrid heuristic is NOT admissible")

### 2.3 Search algorithms: A\* (Algorithm 1), Dijkstra, bidirectional A\*
- **A\*:** an OPEN list (binary heap) and a CLOSED list (set). Nodes on CLOSED are never re-opened, and the search stops when the goal is popped, exactly as in the paper's Algorithm 1. The optional `weight` argument turns plain A\* into the paper's adaptive A\*.
- **Dijkstra:** A\* with h = 0.

In [ ]:
"""Grid search algorithms: (weighted) A* per Algorithm 1, Dijkstra and bidirectional A*.

All searches are written from scratch on top of ``heapq``. Counting convention
(assumption M6): ``expanded`` is the number of nodes moved to the CLOSED list,
``generated`` is the number of distinct nodes ever placed on the OPEN list.
"""


import heapq
import itertools
from dataclasses import dataclass, field


@dataclass
class SearchResult:
    path: list[Cell]            # start ... goal; empty if no path exists
    cost: float                 # accumulated g-cost of the path (1 / sqrt(2) steps)
    expanded: int               # |CLOSED|
    generated: int              # distinct nodes ever opened
    closed: set[Cell] = field(default_factory=set, repr=False)

    @property
    def found(self) -> bool:
        return bool(self.path)


def _reconstruct(parent: dict[Cell, Cell | None], node: Cell) -> list[Cell]:
    path = [node]
    while parent[node] is not None:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path


def astar(grid: GridMap, start: Cell, goal: Cell,
          heuristic: DistanceFn = euclidean, weight: WeightFn | None = None) -> SearchResult:
    """Best-first search with ``f(n) = g(n) + w(n) * h(n)`` (Eq. 1 / Eq. 5).

    Follows the paper's Algorithm 1: a node is finalised when popped from OPEN,
    CLOSED nodes are never re-opened, and the search stops when the goal is popped.
    Ties in ``f`` are broken by smaller ``h`` and then by insertion order
    (assumption M8), which makes every run deterministic.
    """
    if not (grid.is_free(*start) and grid.is_free(*goal)):
        return SearchResult([], float("inf"), 0, 0)

    h_cache: dict[Cell, float] = {}

    def h_term(n: Cell) -> float:
        val = h_cache.get(n)
        if val is None:
            hn = heuristic(n, goal)
            val = hn * weight(n) if weight is not None else hn
            h_cache[n] = val
        return val

    counter = itertools.count()
    g: dict[Cell, float] = {start: 0.0}
    parent: dict[Cell, Cell | None] = {start: None}
    hs = h_term(start)
    open_heap = [(hs, hs, next(counter), start)]
    closed: set[Cell] = set()

    while open_heap:
        _, _, _, current = heapq.heappop(open_heap)
        if current in closed:          # stale heap entry (lazy deletion)
            continue
        if current == goal:
            closed.add(current)
            return SearchResult(_reconstruct(parent, current), g[current],
                                len(closed), len(g), closed)
        closed.add(current)
        g_cur = g[current]
        for nb, step in grid.neighbors(current):
            if nb in closed:
                continue
            g_new = g_cur + step
            if g_new < g.get(nb, float("inf")):
                g[nb] = g_new
                parent[nb] = current
                hn = h_term(nb)
                heapq.heappush(open_heap, (g_new + hn, hn, next(counter), nb))

    return SearchResult([], float("inf"), len(closed), len(g), closed)


def dijkstra(grid: GridMap, start: Cell, goal: Cell) -> SearchResult:
    """Dijkstra's algorithm = A* with h(n) = 0."""
    return astar(grid, start, goal, heuristic=zero)


def bidirectional_astar(grid: GridMap, start: Cell, goal: Cell,
                        heuristic: DistanceFn = euclidean) -> SearchResult:
    """Bidirectional A* (assumption M14).

    A forward search (towards ``goal``) and a backward search (towards ``start``)
    alternate one expansion each. The search stops as soon as a node expanded by
    one side has already been expanded by the other, and the two half-paths are
    joined there. Stopping at the first meeting node (rather than continuing
    until the optimal-termination bound is met) is the textbook simple variant;
    it is fast but may return a slightly longer path, matching the paper's
    observation that BA* paths are longer.
    """
    if not (grid.is_free(*start) and grid.is_free(*goal)):
        return SearchResult([], float("inf"), 0, 0)
    if start == goal:
        return SearchResult([start], 0.0, 1, 1, {start})

    counter = itertools.count()
    sides = []
    for origin, target in ((start, goal), (goal, start)):
        h0 = heuristic(origin, target)
        sides.append({
            "target": target,
            "g": {origin: 0.0},
            "parent": {origin: None},
            "open": [(h0, h0, next(counter), origin)],
            "closed": set(),
        })

    turn = 0
    while sides[0]["open"] and sides[1]["open"]:
        me, other = sides[turn], sides[1 - turn]
        _, _, _, current = heapq.heappop(me["open"])
        if current in me["closed"]:
            continue
        me["closed"].add(current)
        if current in other["closed"]:
            fwd, bwd = sides[0], sides[1]
            first = _reconstruct(fwd["parent"], current)
            second = _reconstruct(bwd["parent"], current)[::-1][1:]
            closed = fwd["closed"] | bwd["closed"]
            cost = fwd["g"][current] + bwd["g"][current]
            generated = len(set(fwd["g"]) | set(bwd["g"]))
            return SearchResult(first + second, cost,
                                len(fwd["closed"]) + len(bwd["closed"]), generated, closed)
        g_cur = me["g"][current]
        for nb, step in grid.neighbors(current):
            if nb in me["closed"]:
                continue
            g_new = g_cur + step
            if g_new < me["g"].get(nb, float("inf")):
                me["g"][nb] = g_new
                me["parent"][nb] = current
                hn = heuristic(nb, me["target"])
                heapq.heappush(me["open"], (g_new + hn, hn, next(counter), nb))
        turn = 1 - turn

    expanded = len(sides[0]["closed"]) + len(sides[1]["closed"])
    return SearchResult([], float("inf"), expanded, len(set(sides[0]["g"]) | set(sides[1]["g"])))

### 2.4 Redundant point deletion (Contribution 3, Sec. 3.3, Fig. 4)
A three-node window (P*i*, P*i*+1, P*i*+2) moves along the raw path:
- **Step 2:** if the three nodes are collinear, delete the middle one.
- **Step 3:** otherwise, connect P*i* to P*i*+2. If that straight line touches no obstacle cell (exact *line-of-sight* test), delete the middle node; if it does, keep it.

In [ ]:
"""Redundant point deletion strategy (paper Sec. 3.3, Fig. 4).

Path nodes are cell centres. A straight segment between two nodes is safe only
if it does not touch any obstacle cell. Cells are treated as closed squares
``[x - 0.5, x + 0.5] x [y - 0.5, y + 0.5]``, so a segment that merely grazes an
obstacle corner is rejected (assumption M9: conservative, no corner cutting -
consistent with the search's no-corner-cutting rule).
"""


import math


_EPS = 1e-9


def segment_cells(a: Cell, b: Cell) -> list[Cell]:
    """All grid cells whose closed square intersects the segment a-b (exact supercover)."""
    (x0, y0), (x1, y1) = a, b
    cells: list[Cell] = []
    if x0 == x1:
        for y in range(min(y0, y1), max(y0, y1) + 1):
            cells.append((x0, y))
        return cells
    if x0 > x1:
        (x0, y0), (x1, y1) = (x1, y1), (x0, y0)
    slope = (y1 - y0) / (x1 - x0)
    for cx in range(x0, x1 + 1):
        # Portion of the segment lying inside column cx (|x - cx| <= 0.5).
        xa = max(cx - 0.5, x0)
        xb = min(cx + 0.5, x1)
        ya = y0 + slope * (xa - x0)
        yb = y0 + slope * (xb - x0)
        lo, hi = min(ya, yb), max(ya, yb)
        # Rows r whose interval [r - 0.5, r + 0.5] meets [lo, hi].
        r_lo = math.ceil(lo - 0.5 - _EPS)
        r_hi = math.floor(hi + 0.5 + _EPS)
        for cy in range(r_lo, r_hi + 1):
            cells.append((cx, cy))
    # Column boundaries are closed too: a segment passing exactly through the
    # boundary x = cx + 0.5 already enumerates both columns above.
    return cells


def line_of_sight(grid: GridMap, a: Cell, b: Cell) -> bool:
    """True if the straight segment between cell centres a and b is obstacle-free."""
    for x, y in segment_cells(a, b):
        if not grid.is_free(x, y):
            return False
    return True


def is_collinear(p: Cell, q: Cell, r: Cell) -> bool:
    """Collinearity test via the 2-D cross product (Step 2)."""
    return (q[0] - p[0]) * (r[1] - p[1]) - (q[1] - p[1]) * (r[0] - p[0]) == 0


def delete_redundant_points(grid: GridMap, path: list[Cell], mode: str = "anchor") -> list[Cell]:
    """Remove redundant nodes from a grid path (Steps 1-4 of Sec. 3.3).

    Walk the path with a three-node window (P_i, P_i+1, P_i+2) anchored at P_i:
      * Step 2 - if the three nodes are collinear and the segment is free,
        delete the middle node;
      * Step 3 - otherwise connect P_i to P_i+2 and delete the middle node if
        that line does not cross an obstacle, else keep it and advance the anchor.
    After a deletion the anchor stays put and the window slides to the next
    node (assumption M10), so the anchor keeps "pulling the string" until line
    of sight is lost. Step 4 joins the kept nodes into the new path.

    ``mode="single-pass"`` is the alternative, more literal reading used as a
    sensitivity check: the window is never re-anchored, i.e. after deleting the
    middle node the window jumps to the next triple (P_i+2, P_i+3, P_i+4), so
    each node is examined once and at most every other node can be removed.
    """
    if mode not in ("anchor", "single-pass"):
        raise ValueError("mode must be 'anchor' or 'single-pass'")
    if len(path) <= 2:
        return list(path)
    kept = [path[0]]
    anchor = path[0]
    i = 1
    while i < len(path) - 1:
        middle, nxt = path[i], path[i + 1]
        # Step 2: a collinear triple is free by construction (anchor->middle was
        # already verified and middle->nxt is a free grid step). Step 3: otherwise
        # the shortcut anchor->nxt needs an explicit line-of-sight test.
        removable = is_collinear(anchor, middle, nxt) or line_of_sight(grid, anchor, nxt)
        if not removable:
            kept.append(middle)                    # Step 3: shortcut blocked, keep node
            anchor = middle
            i += 1
        elif mode == "anchor":
            i += 1                                 # keep the anchor, test the next node
        else:
            if i + 1 < len(path) - 1:
                kept.append(nxt)                   # window jumps past the deleted node
            anchor = nxt
            i += 2
    kept.append(path[-1])
    return kept

### 2.5 Evaluation metrics (as in Tables 1–5)
Path length (m), expanded nodes (size of CLOSED), number of corners with heading change ≥ 45°, and planning time.

In [ ]:
"""Evaluation metrics used in the paper's Tables 1-5."""


import math


def path_length(path: list[Cell], cell_size: float = 1.0) -> float:
    """Euclidean polyline length in metres (assumption M16)."""
    return cell_size * sum(math.dist(path[i], path[i + 1]) for i in range(len(path) - 1))


def turn_angles(path: list[Cell]) -> list[float]:
    """Absolute heading change (degrees) at every interior node."""
    angles = []
    for i in range(1, len(path) - 1):
        ax, ay = path[i][0] - path[i - 1][0], path[i][1] - path[i - 1][1]
        bx, by = path[i + 1][0] - path[i][0], path[i + 1][1] - path[i][1]
        cross = ax * by - ay * bx
        dot = ax * bx + ay * by
        angles.append(abs(math.degrees(math.atan2(cross, dot))))
    return angles


def count_sharp_turns(path: list[Cell], threshold_deg: float = 45.0) -> int:
    """Number of corners whose heading change is >= threshold (assumption M7)."""
    return sum(1 for a in turn_angles(path) if a >= threshold_deg - 1e-9)

### 2.6 Planners: every combination needed for the experiments
`A*` is the baseline. The paper's three improvements can each be switched on alone (the ablation of Table 1) or all together (**Adaptive A\***).

In [ ]:
"""Named planners used in the experiments, built from the three switches of the paper.

Every planner returns a :class:`PlanResult` with the raw grid path, the final
path (after redundant point deletion when enabled) and the four paper metrics.

=====================  ===============  ==============  =============  =======
Planner                heuristic h(n)   weight w(n)     pruning        paper
=====================  ===============  ==============  =============  =======
A*                     Euclidean        1               no             Tab 1-5
Dijkstra               0                -               no             Tab 2-4
Bidirectional A*       Euclidean        1               no             Tab 2-4
AEF (isolated)         Euclidean        Eq. (6)         no             Tab 1
DMO (isolated)         hybrid Eq. (7)   1               no             Tab 1
RPD (isolated)         Euclidean        1               yes            Tab 1
AEF+DMO (cumulative)   hybrid           Eq. (6)         no             extra
Adaptive A*            hybrid           Eq. (6)         yes            Tab 1-5
A*+Octile+RPD          octile           1               yes            extra
... (single-pass)      as above         as above        single-pass    extra
=====================  ===============  ==============  =============  =======
"""


import gc
import statistics
import time
from dataclasses import dataclass
from typing import Callable


@dataclass
class PlanResult:
    planner: str
    raw_path: list[Cell]
    path: list[Cell]
    expanded: int
    generated: int
    raw_cost: float
    length: float
    corners: int
    closed: set[Cell]

    @property
    def found(self) -> bool:
        return bool(self.path)


@dataclass(frozen=True)
class PlannerSpec:
    name: str
    search: str = "astar"          # astar | dijkstra | bidirectional
    heuristic: str = "euclidean"   # euclidean | hybrid | octile
    adaptive: bool = False         # use the adaptive weight of Eq. (6)
    prune: bool = False            # apply redundant point deletion (Sec. 3.3)
    prune_mode: str = "anchor"     # anchor | single-pass (sensitivity, assumption M10)

    def search_only(self, grid: GridMap, start: Cell, goal: Cell) -> SearchResult:
        h = {"euclidean": euclidean, "hybrid": hybrid, "octile": octile}[self.heuristic]
        if self.search == "dijkstra":
            return dijkstra(grid, start, goal)
        if self.search == "bidirectional":
            return bidirectional_astar(grid, start, goal, heuristic=h)
        weight = adaptive_weight(start, goal) if self.adaptive else None
        return astar(grid, start, goal, heuristic=h, weight=weight)

    def plan(self, grid: GridMap, start: Cell, goal: Cell) -> PlanResult:
        res = self.search_only(grid, start, goal)
        final = delete_redundant_points(grid, res.path, self.prune_mode) if (self.prune and res.found) else res.path
        return PlanResult(
            planner=self.name, raw_path=res.path, path=final,
            expanded=res.expanded, generated=res.generated, raw_cost=res.cost,
            length=path_length(final) if final else float("nan"),
            corners=count_sharp_turns(final) if final else -1,
            closed=res.closed,
        )


PLANNERS: dict[str, PlannerSpec] = {p.name: p for p in [
    PlannerSpec("A*"),
    PlannerSpec("Dijkstra", search="dijkstra"),
    PlannerSpec("Bidirectional A*", search="bidirectional"),
    PlannerSpec("Adaptive estimation function", adaptive=True),
    PlannerSpec("Distance model optimization", heuristic="hybrid"),
    PlannerSpec("Redundant point deletion", prune=True),
    PlannerSpec("AEF+DMO (no pruning)", heuristic="hybrid", adaptive=True),
    PlannerSpec("Adaptive A*", heuristic="hybrid", adaptive=True, prune=True),
    PlannerSpec("A*+Octile+RPD", heuristic="octile", prune=True),
    PlannerSpec("Redundant point deletion (single-pass)", prune=True, prune_mode="single-pass"),
    PlannerSpec("Adaptive A* (single-pass pruning)", heuristic="hybrid", adaptive=True,
                prune=True, prune_mode="single-pass"),
]}


def time_planner(spec: PlannerSpec, grid: GridMap, start: Cell, goal: Cell,
                 repeats: int = 30, warmup: int = 2) -> dict[str, float]:
    """Wall-clock time of search + pruning (assumption M11/M12).

    Returns median, inter-quartile range, mean and std over ``repeats`` runs,
    measured with ``time.perf_counter`` and the garbage collector disabled.
    """
    run: Callable[[], PlanResult] = lambda: spec.plan(grid, start, goal)
    for _ in range(warmup):
        run()
    samples = []
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        for _ in range(repeats):
            t0 = time.perf_counter()
            run()
            samples.append(time.perf_counter() - t0)
    finally:
        if gc_was_enabled:
            gc.enable()
    q = statistics.quantiles(samples, n=4) if len(samples) >= 2 else [samples[0]] * 3
    return {
        "time_median": statistics.median(samples),
        "time_iqr": q[2] - q[0],
        "time_mean": statistics.fmean(samples),
        "time_std": statistics.pstdev(samples),
        "repeats": repeats,
    }

## 3. Sanity checks
These tests confirm the implementation behaves correctly before any experiment is run.

In [ ]:
def check(name, condition):
    print(("PASS " if condition else "FAIL ") + name)
    assert condition, name

e = GridMap(np.zeros((10, 10), dtype=bool))
check("A* on an empty grid finds the exact diagonal cost 9*sqrt(2)",
      abs(astar(e, (0, 0), (9, 9)).cost - 9 * math.sqrt(2)) < 1e-9)
occ = np.zeros((3, 3), dtype=bool); occ[0, 1] = occ[1, 0] = True
check("no diagonal corner cutting between touching obstacles",
      (1, 1) not in [n for n, _ in GridMap(occ).neighbors((0, 0))])
occ = np.zeros((5, 5), dtype=bool); occ[2, 2] = True; g5 = GridMap(occ)
check("line of sight is blocked through an obstacle", not line_of_sight(g5, (0, 0), (4, 4)))
check("line of sight is free along an empty row", line_of_sight(g5, (0, 0), (4, 0)))
check("pruning straightens a staircase into one segment",
      delete_redundant_points(e, [(0, 0), (1, 0), (2, 0), (3, 1), (4, 2), (5, 3)]) == [(0, 0), (5, 3)])
check("a 45-degree turn is counted as a corner", count_sharp_turns([(0, 0), (1, 0), (2, 1)]) == 1)
check("adaptive weight = ln(1+e) at start and ln 2 at goal",
      abs(adaptive_weight((0, 0), (10, 0))((0, 0)) - math.log(1 + math.e)) < 1e-12
      and abs(adaptive_weight((0, 0), (10, 0))((10, 0)) - math.log(2)) < 1e-12)

## 4. Maps

The paper shows its three maps **only as figures** (30×30 irregular outdoor, 60×60 maze/corridor, 100×100 array-like). No map data is published.

We therefore use **proxy maps** of the same size and scene type:
- Thousands of seeded candidate maps were generated, and the one whose **Dijkstra / A\* statistics** best match the paper's was kept (repository script `scripts/calibrate_maps.py`).
- The proposed method was never used during calibration, so the maps are not tuned in its favour.
- The generator code is below. Each map is rebuilt exactly from its recorded seed.

In [ ]:
"""Procedural generators for the three scene types of the paper (Sec. 4.1, Fig. 5).

The original obstacle matrices are not published (missing item M1), so the
experiments use *proxy maps*: maps of the same size and scene type whose
parameters are calibrated (``scripts/calibrate_maps.py``) so that Dijkstra and
A* reproduce the paper's published search statistics as closely as possible.

* ``irregular``  - map1, 30 x 30, irregular outdoor scene (random blobs);
* ``maze``       - map2, 60 x 60, structured indoor scene (maze + corridors);
* ``array``      - map3, 100 x 100, array-like scene (PV panel rows / racks).

All generators take an explicit ``numpy.random.Generator`` so every map is
reproducible from its seed.
"""


from collections import deque

import numpy as np


# Start/goal pairs used in the paper (1-based paper coordinates).
PAPER_POINTS: dict[str, list[tuple[int, int]]] = {
    "map1": [(2, 2), (28, 28), (4, 24), (29, 2)],
    "map2": [(2, 2), (58, 58), (4, 54), (59, 2)],
    "map3": [(2, 2), (99, 99)],
}
MAP_SIZES = {"map1": 30, "map2": 60, "map3": 100}
MAP_KINDS = {"map1": "irregular", "map2": "maze", "map3": "array"}


# --------------------------------------------------------------- generators
def irregular(n: int, rng: np.random.Generator, n_blobs: int = 18,
              blob_cells: tuple[int, int] = (6, 30)) -> np.ndarray:
    """Irregular outdoor obstacles: blobs grown by random cell accretion."""
    occ = np.zeros((n, n), dtype=bool)
    for _ in range(n_blobs):
        size = int(rng.integers(blob_cells[0], blob_cells[1] + 1))
        blob = [(int(rng.integers(0, n)), int(rng.integers(0, n)))]
        occ[blob[0][1], blob[0][0]] = True
        attempts = 0
        while len(blob) < size and attempts < 50 * size:   # a boxed-in blob just stops growing
            attempts += 1
            x, y = blob[int(rng.integers(0, len(blob)))]
            dx, dy = ((1, 0), (-1, 0), (0, 1), (0, -1))[int(rng.integers(0, 4))]
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and not occ[ny, nx]:
                occ[ny, nx] = True
                blob.append((nx, ny))
    return occ


def maze(n: int, rng: np.random.Generator, corridor: int = 3, wall: int = 2,
         loop_prob: float = 0.15) -> np.ndarray:
    """Structured indoor scene: a randomised-DFS maze with corridors of width
    ``corridor`` and walls of thickness ``wall``; a fraction ``loop_prob`` of the
    remaining internal walls is opened to create alternative routes."""
    pitch = corridor + wall
    m = max(2, (n - wall) // pitch)                    # coarse cells per side
    occ = np.ones((n, n), dtype=bool)
    offset = (n - (m * pitch + wall)) // 2

    def room_origin(i: int, j: int) -> tuple[int, int]:
        return offset + wall + i * pitch, offset + wall + j * pitch

    for i in range(m):
        for j in range(m):
            x0, y0 = room_origin(i, j)
            occ[y0:y0 + corridor, x0:x0 + corridor] = False

    def open_between(a: tuple[int, int], b: tuple[int, int]) -> None:
        (i0, j0), (i1, j1) = a, b
        x0, y0 = room_origin(min(i0, i1), min(j0, j1))
        if i0 != i1:   # horizontal neighbours: knock out the vertical wall
            occ[y0:y0 + corridor, x0 + corridor:x0 + pitch] = False
        else:
            occ[y0 + corridor:y0 + pitch, x0:x0 + corridor] = False

    visited = np.zeros((m, m), dtype=bool)
    stack = [(0, 0)]
    visited[0, 0] = True
    walls_left = []
    while stack:
        i, j = stack[-1]
        nbrs = [(i + di, j + dj) for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1))
                if 0 <= i + di < m and 0 <= j + dj < m and not visited[i + di, j + dj]]
        if not nbrs:
            stack.pop()
            continue
        k = nbrs[int(rng.integers(0, len(nbrs)))]
        open_between((i, j), k)
        visited[k] = True
        stack.append(k)
    for i in range(m):
        for j in range(m):
            for di, dj in ((1, 0), (0, 1)):
                if i + di < m and j + dj < m:
                    walls_left.append(((i, j), (i + di, j + dj)))
    for a, b in walls_left:
        if rng.random() < loop_prob:
            open_between(a, b)
    return occ


def array_field(n: int, rng: np.random.Generator, panel_h: int = 2, aisle: int = 3,
                panel_len: tuple[int, int] = (8, 16), gap: tuple[int, int] = (1, 3),
                missing_prob: float = 0.08, clutter: int = 20) -> np.ndarray:
    """Array-like scene: staggered rows of rectangular panels (PV strings,
    warehouse racks) separated by aisles, with random gaps, missing panels and
    a little random clutter."""
    occ = np.zeros((n, n), dtype=bool)
    y = aisle
    while y + panel_h < n - 1:
        x = int(rng.integers(0, panel_len[1]))
        while x < n:
            length = int(rng.integers(panel_len[0], panel_len[1] + 1))
            if rng.random() >= missing_prob:
                occ[y:y + panel_h, x:min(n, x + length)] = True
            x += length + int(rng.integers(gap[0], gap[1] + 1))
        y += panel_h + aisle
    for _ in range(clutter):
        cx, cy = int(rng.integers(0, n)), int(rng.integers(0, n))
        occ[cy:cy + int(rng.integers(1, 3)), cx:cx + int(rng.integers(1, 3))] = True
    return occ


GENERATORS = {"irregular": irregular, "maze": maze, "array": array_field}


# ----------------------------------------------------------------- helpers
def clear_points(occ: np.ndarray, cells: list[Cell], radius: int = 1) -> None:
    """Free the given cells and their (2r+1)^2 neighbourhood in place."""
    h, w = occ.shape
    for x, y in cells:
        occ[max(0, y - radius):min(h, y + radius + 1), max(0, x - radius):min(w, x + radius + 1)] = False


def connected(grid: GridMap, cells: list[Cell]) -> bool:
    """True if all cells lie in one 8-connected free component (BFS)."""
    if not all(grid.is_free(*c) for c in cells):
        return False
    seen = {cells[0]}
    queue = deque([cells[0]])
    while queue:
        cur = queue.popleft()
        for nb, _ in grid.neighbors(cur):
            if nb not in seen:
                seen.add(nb)
                queue.append(nb)
    return all(c in seen for c in cells)


def largest_component(grid: GridMap) -> set[Cell]:
    """Cells of the largest 8-connected free component."""
    unseen = {(x, y) for y in range(grid.height) for x in range(grid.width) if grid.is_free(x, y)}
    best: set[Cell] = set()
    while unseen:
        seed = unseen.pop()
        comp = {seed}
        queue = deque([seed])
        while queue:
            for nb, _ in grid.neighbors(queue.popleft()):
                if nb in unseen:
                    unseen.discard(nb)
                    comp.add(nb)
                    queue.append(nb)
        if len(comp) > len(best):
            best = comp
    return best


def carve_access(occ: np.ndarray, cells: list[Cell]) -> None:
    """Connect each required cell to the main free region with an L-shaped,
    1-cell-wide corridor (needed when a start/goal falls inside a wall)."""
    comp = largest_component(GridMap(occ))
    for x, y in cells:
        if (x, y) in comp:
            continue
        tx, ty = min(comp, key=lambda c: (c[0] - x) ** 2 + (c[1] - y) ** 2)
        occ[y, min(x, tx):max(x, tx) + 1] = False
        occ[min(y, ty):max(y, ty) + 1, tx] = False
        comp = largest_component(GridMap(occ))


def make_map(map_id: str, seed: int, **params) -> GridMap | None:
    """Generate a proxy map for ``map_id`` ("map1" | "map2" | "map3").

    The paper's start/goal cells are freed and, if necessary, linked to the
    main free region. Returns ``None`` if they are still not mutually reachable.
    """
    n = MAP_SIZES[map_id]
    rng = np.random.default_rng(seed)
    occ = GENERATORS[MAP_KINDS[map_id]](n, rng, **params)
    cells = [from_paper(*p) for p in PAPER_POINTS[map_id]]
    clear_points(occ, cells, radius=1)
    carve_access(occ, cells)
    grid = GridMap(occ, name=f"{map_id}_seed{seed}")
    return grid if connected(grid, cells) else None

In [ ]:
# Calibrated generator parameters and seeds (from maps/calibration.json)
CALIBRATION = {
    "map1": {
        "seed": 1796752687,
        "params": {
            "n_blobs": 17,
            "blob_cells": [
                4,
                35
            ]
        }
    },
    "map2": {
        "seed": 2048598178,
        "params": {
            "corridor": 4,
            "wall": 3,
            "loop_prob": 0.204
        }
    },
    "map3": {
        "seed": 694909934,
        "params": {
            "panel_h": 2,
            "aisle": 2,
            "panel_len": [
                5,
                7
            ],
            "gap": [
                2,
                4
            ],
            "missing_prob": 0.296,
            "clutter": 6
        }
    }
}

def build_proxy_map(map_id):
    c = CALIBRATION[map_id]
    params = {k: tuple(v) if isinstance(v, list) else v for k, v in c["params"].items()}
    grid = make_map(map_id, c["seed"], **params)
    grid.name = map_id
    return grid

MAPS = {"30x30": build_proxy_map("map1"), "60x60": build_proxy_map("map2"), "100x100": build_proxy_map("map3")}
for size, grid in MAPS.items():
    print(size, grid)

In [ ]:
# How close are the proxy maps to the paper? (baseline statistics only)
rows = []
for size, goal, paper in (("30x30", (28, 28), (632, 40.30, 130)), ("60x60", (58, 58), (1740, 92.10, 716)),
                          ("100x100", (99, 99), (7520, 146.41, 2191))):
    s, t = from_paper(2, 2), from_paper(*goal)
    d, a = dijkstra(MAPS[size], s, t), astar(MAPS[size], s, t)
    rows.append({"map": size, "Dijkstra nodes (paper)": paper[0], "Dijkstra nodes (proxy)": d.expanded,
                 "optimal length (paper)": paper[1], "optimal length (proxy)": round(d.cost, 2),
                 "A* nodes (paper)": paper[2], "A* nodes (proxy)": a.expanded})
display(pd.DataFrame(rows).set_index("map"))

In [ ]:
# Palette and a helper that draws a map with explored cells and a path
INK, INK2, MUTED, SURFACE, GRID_LINE = "#0b0b0b", "#52514e", "#8a8984", "#fcfcfb", "#e4e3df"
OBSTACLE, EXPLORED, BLUE, ORANGE = "#3a3936", "#cde2fb", "#2a78d6", "#eb6834"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9.5, "axes.edgecolor": GRID_LINE,
                     "figure.facecolor": "white", "axes.facecolor": SURFACE})

def draw(ax, grid, start, goal, planner=None, title="", color=BLUE):
    img = np.zeros(grid.occ.shape, dtype=int)
    res = None
    if planner:
        res = PLANNERS[planner].plan(grid, from_paper(*start), from_paper(*goal))
        for x, y in res.closed:
            img[y, x] = 1
    img[grid.occ] = 2
    ax.imshow(img, cmap=ListedColormap([SURFACE, EXPLORED, OBSTACLE]), vmin=0, vmax=2, origin="lower",
              interpolation="nearest", extent=(0.5, grid.width + 0.5, 0.5, grid.height + 0.5))
    if res:
        xs, ys = [c[0] + 1 for c in res.path], [c[1] + 1 for c in res.path]
        ax.plot(xs, ys, color=color, lw=1.8)
        if PLANNERS[planner].prune:
            ax.plot(xs, ys, "o", ms=2.8, color=color)
        title += f"\nL={res.length:.2f} m, N={res.expanded}, C={res.corners}"
    ax.plot(*start, "s", ms=6, color=INK)
    ax.plot(*goal, "*", ms=10, color=INK, mec=SURFACE)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (size, goal) in zip(axes, (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99)))):
    draw(ax, MAPS[size], (2, 2), goal, title=f"Proxy map {size}")
plt.show()

## 5. The paper's published results (Tables 1–5)

In [ ]:
PUBLISHED_CSV = """
table,map,start,goal,algorithm,time_s,length_m,expanded_nodes,corners_ge45
T1,30x30,2;2,28;28,A*,0.04,40.41,130,9
T1,30x30,2;2,28;28,Adaptive estimation function,0.03,40.20,118,13
T1,30x30,2;2,28;28,Distance model optimization,0.02,41.45,45,9
T1,30x30,2;2,28;28,Redundant point deletion,0.02,38.74,130,5
T1,60x60,2;2,58;58,A*,0.18,92.30,716,12
T1,60x60,2;2,58;58,Adaptive estimation function,0.09,92.05,412,12
T1,60x60,2;2,58;58,Distance model optimization,0.12,91.08,691,10
T1,60x60,2;2,58;58,Redundant point deletion,0.12,87.49,691,7
T2,30x30,2;2,28;28,A*,0.04,40.41,130,9
T2,30x30,2;2,28;28,Dijkstra [28],0.14,40.30,632,11
T2,30x30,2;2,28;28,Chen et al. [29],0.03,41.69,129,9
T2,30x30,2;2,28;28,Bidirectional A* [30],0.03,42.65,156,9
T2,30x30,2;2,28;28,Zhang et al. [31],0.04,40.29,98,11
T2,30x30,2;2,28;28,Adaptive A*,0.01,39.72,36,7
T3,60x60,2;2,58;58,A*,0.18,92.30,716,12
T3,60x60,2;2,58;58,Dijkstra [28],0.39,92.10,1740,13
T3,60x60,2;2,58;58,Chen et al. [29],0.12,92.08,696,12
T3,60x60,2;2,58;58,Bidirectional A* [30],0.18,93.91,704,12
T3,60x60,2;2,58;58,Zhang et al. [31],0.24,92.11,523,10
T3,60x60,2;2,58;58,Adaptive A*,0.06,90.89,268,8
T4,100x100,2;2,99;99,A*,0.70,146.56,2191,34
T4,100x100,2;2,99;99,Dijkstra [28],1.93,146.41,7520,31
T4,100x100,2;2,99;99,Chen et al. [29],0.52,146.56,2168,34
T4,100x100,2;2,99;99,Bidirectional A* [30],0.38,148.38,1126,32
T4,100x100,2;2,99;99,Zhang et al. [31],0.45,146.56,891,30
T4,100x100,2;2,99;99,Adaptive A*,0.25,140.67,113,14
T5,30x30,28;28,2;2,A*,0.03,40.30,202,13
T5,30x30,28;28,2;2,Adaptive A*,0.01,39.92,43,6
T5,30x30,4;24,29;2,A*,0.04,38.21,250,14
T5,30x30,4;24,29;2,Adaptive A*,0.01,37.68,33,5
T5,60x60,58;58,2;2,A*,0.25,92.08,898,19
T5,60x60,58;58,2;2,Adaptive A*,0.04,88.81,123,7
T5,60x60,4;54,59;2,A*,0.41,118.49,1159,27
T5,60x60,4;54,59;2,Adaptive A*,0.13,114.96,478,13
"""
PUBLISHED = pd.read_csv(io.StringIO(PUBLISHED_CSV.strip()))
PUBLISHED["algorithm"] = PUBLISHED["algorithm"].replace({"Dijkstra [28]": "Dijkstra",
                                                          "Bidirectional A* [30]": "Bidirectional A*"})
display(PUBLISHED)

## 6. Experiments

Each experiment runs on the proxy maps with the paper's start/goal points.
- **Length, nodes and corners** are deterministic, identical on every run and every machine.
- **Time** is the median of repeated runs. The paper used MATLAB and we use Python, so absolute times are not comparable. We compare the **change relative to A\*** instead.

In [ ]:
METRICS = ["time_s", "length_m", "expanded_nodes", "corners_ge45"]
PUBLISHED_ONLY = {"Chen et al. [29]", "Zhang et al. [31]"}   # not described in the paper -> published numbers only

def run_case(name, size, start, goal):
    spec = PLANNERS[name]
    s, t = from_paper(*start), from_paper(*goal)
    res = spec.plan(MAPS[size], s, t)
    timing = time_planner(spec, MAPS[size], s, t, repeats=TIMING_REPEATS)
    return {"algorithm": name, "time_s": timing["time_median"], "length_m": res.length,
            "expanded_nodes": res.expanded, "corners_ge45": res.corners, "raw_cost": res.raw_cost}

def relative(df):
    """Change relative to the A* row: metres for length, percent for the rest."""
    base = df[df["algorithm"] == "A*"].iloc[0]
    out = df[["algorithm"]].copy()
    for m in METRICS:
        out[m] = df[m] - base[m] if m == "length_m" else 100 * (df[m] - base[m]) / base[m]
    return out

def experiment(table, size, start, goal, algorithms):
    """Run one experiment and show paper vs ours (absolute values and change vs A*)."""
    ours = pd.DataFrame([run_case(a, size, start, goal) for a in algorithms])
    key = f"{start[0]};{start[1]}"
    pub = PUBLISHED[(PUBLISHED["table"] == table) & (PUBLISHED["map"] == size)
                    & (PUBLISHED["start"] == key) & (PUBLISHED["goal"] == f"{goal[0]};{goal[1]}")]
    merged = pub[["algorithm"] + METRICS].merge(ours[["algorithm"] + METRICS], on="algorithm",
                                                 how="outer", suffixes=(" paper", " ours"))
    order = list(dict.fromkeys(list(pub["algorithm"]) + list(ours["algorithm"])))
    merged = merged.set_index("algorithm").loc[order]
    absolute = pd.DataFrame(index=merged.index)
    for m in METRICS:
        absolute[f"{m} paper"] = merged[f"{m} paper"]
        absolute[f"{m} ours"] = merged[f"{m} ours"]
        if m != "time_s":
            absolute[f"{m} diff"] = merged[f"{m} ours"] - merged[f"{m} paper"]
    rel_p = relative(pub.reset_index(drop=True)).set_index("algorithm").add_suffix(" paper")
    rel_o = relative(ours).set_index("algorithm").add_suffix(" ours")
    rel = rel_p.join(rel_o, how="outer").reindex(order).drop(index="A*")
    rel = rel[[f"{m} {w}" for m in METRICS for w in ("paper", "ours")]]
    display(Markdown(f"**{table} — {size}, start {start} → goal {goal}** (absolute values)"))
    display(absolute.style.format(precision=4, na_rep="—"))
    display(Markdown("Change relative to A\\* (length in m, others in %; negative = better):"))
    display(rel.style.format("{:+.1f}", na_rep="—"))
    return ours.assign(table=table, map=size, start=str(start), goal=str(goal))

ALL_RESULTS = []

### 6.1 Table 1 — Ablation of the three improvements
- **AEF** = adaptive estimation function only.
- **DMO** = hybrid distance model only.
- **RPD** = redundant point deletion only.

We also add the full combination and a single-pass pruning variant.

In [ ]:
ABLATION = ["A*", "Adaptive estimation function", "Distance model optimization", "Redundant point deletion",
            "AEF+DMO (no pruning)", "Adaptive A*"]
ALL_RESULTS.append(experiment("T1", "30x30", (2, 2), (28, 28), ABLATION))

In [ ]:
ALL_RESULTS.append(experiment("T1", "60x60", (2, 2), (58, 58), ABLATION))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (name, title) in zip(axes, [("A*", "(a) A*"), ("Adaptive estimation function", "(b) Adaptive estimation"),
                                    ("Distance model optimization", "(c) Distance model"),
                                    ("Redundant point deletion", "(d) Redundant point deletion")]):
    draw(ax, MAPS["60x60"], (2, 2), (58, 58), name, title)
fig.suptitle("Ablation on the 60x60 map (cf. paper Fig. 7) - explored cells shaded", color=INK2)
plt.show()

### 6.2 Tables 2–4 — Comparison with other algorithms
Chen et al. [29] and Zhang et al. [31] are not described in the paper, so they appear with their **published numbers only**.

**A\* + pruning** and **A\* + octile + pruning** are extra baselines we added for a fair comparison.

In [ ]:
COMPARISON = ["A*", "Dijkstra", "Bidirectional A*", "Adaptive A*", "Redundant point deletion", "A*+Octile+RPD"]
ALL_RESULTS.append(experiment("T2", "30x30", (2, 2), (28, 28), COMPARISON))

In [ ]:
ALL_RESULTS.append(experiment("T3", "60x60", (2, 2), (58, 58), COMPARISON))

In [ ]:
ALL_RESULTS.append(experiment("T4", "100x100", (2, 2), (99, 99), COMPARISON))

In [ ]:
for size, goal in (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99))):
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
    for ax, (name, title) in zip(axes, [("A*", "(a) A*"), ("Dijkstra", "(b) Dijkstra"),
                                        ("Bidirectional A*", "(c) Bidirectional A*"),
                                        ("Redundant point deletion", "(d) A* + pruning"),
                                        ("Adaptive A*", "(e) Adaptive A*")]):
        draw(ax, MAPS[size], (2, 2), goal, name, title, ORANGE if name == "Adaptive A*" else BLUE)
    fig.suptitle(f"Algorithms on the {size} map (cf. paper Figs. 8-10)", color=INK2)
    plt.show()

### 6.3 Table 5 — Different start and end points

In [ ]:
for size, start, goal in (("30x30", (28, 28), (2, 2)), ("30x30", (4, 24), (29, 2)),
                          ("60x60", (58, 58), (2, 2)), ("60x60", (4, 54), (59, 2))):
    ALL_RESULTS.append(experiment("T5", size, start, goal, ["A*", "Adaptive A*"]))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for col, (size, s, g) in enumerate((("30x30", (28, 28), (2, 2)), ("30x30", (4, 24), (29, 2)),
                                    ("60x60", (58, 58), (2, 2)), ("60x60", (4, 54), (59, 2)))):
    draw(axes[0, col], MAPS[size], s, g, "A*", f"A* {s}->{g}")
    draw(axes[1, col], MAPS[size], s, g, "Adaptive A*", f"Adaptive A* {s}->{g}", ORANGE)
fig.suptitle("Different start and end points (cf. paper Figs. 11-12)", color=INK2)
plt.show()

## 7. Paired timing study (planning-time metric)
A single run takes well under a millisecond, so one-off timings are noisy. Here A\* and Adaptive A\* are run **alternately**, many times, and we report the median of the paired ratios with a bootstrap 95% confidence interval.

In [ ]:
def paired_timing(size, goal, pairs=TIMING_PAIRS, seed=0):
    grid, s, t = MAPS[size], from_paper(2, 2), from_paper(*goal)
    a_spec, b_spec = PLANNERS["A*"], PLANNERS["Adaptive A*"]
    for _ in range(5):
        a_spec.plan(grid, s, t); b_spec.plan(grid, s, t)
    ta, tb = [], []
    gc.disable()
    try:
        for i in range(pairs):
            for spec in ((a_spec, b_spec) if i % 2 == 0 else (b_spec, a_spec)):
                t0 = time.perf_counter(); spec.plan(grid, s, t)
                (ta if spec is a_spec else tb).append(time.perf_counter() - t0)
    finally:
        gc.enable()
    ratio = np.array(tb) / np.array(ta)
    rng = np.random.default_rng(seed)
    lo, hi = np.percentile([np.median(rng.choice(ratio, ratio.size)) for _ in range(2000)], [2.5, 97.5])
    return {"map": size, "A* median (ms)": 1e3 * np.median(ta), "Adaptive A* median (ms)": 1e3 * np.median(tb),
            "ratio": np.median(ratio), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "time reduction ours (%)": 100 * (1 - np.median(ratio))}

TIMING = pd.DataFrame([paired_timing(s, g) for s, g in (("30x30", (28, 28)), ("60x60", (58, 58)),
                                                        ("100x100", (99, 99)))])
TIMING["time reduction paper (%)"] = [75.0, 66.7, 64.3]
display(TIMING.set_index("map").round(3))

## 8. Headline comparison — published vs. reproduced (assignment table)

In [ ]:
RESULTS = pd.concat(ALL_RESULTS, ignore_index=True)

def mean_reduction(df_rows, metric):
    vals = []
    for size in ("30x30", "60x60", "100x100"):
        r = df_rows[df_rows["map"] == size].set_index("algorithm")
        a, b = r.loc["A*", metric], r.loc["Adaptive A*", metric]
        vals.append(a - b if metric == "length_m" else 100 * (a - b) / a)
    return np.mean(vals), vals

ours_tab = RESULTS[RESULTS["table"].isin(["T2", "T3", "T4"])]
paper_tab = PUBLISHED[PUBLISHED["table"].isin(["T2", "T3", "T4"])]
rows = []
for label, metric, unit in (("Metric 1: planning-time reduction vs A*", "time_s", "%"),
                            ("Metric 2: path-length reduction vs A*", "length_m", "m"),
                            ("Metric 3: expanded-node reduction vs A*", "expanded_nodes", "%"),
                            ("Metric 4: reduction of corners >= 45 deg vs A*", "corners_ge45", "%")):
    p, p_each = mean_reduction(paper_tab, metric)
    if metric == "time_s":   # paired timing study is the reliable time measurement
        o_each = list(TIMING["time reduction ours (%)"]); o = float(np.mean(o_each))
    else:
        o, o_each = mean_reduction(ours_tab, metric)
    fmt = (lambda v: f"{v:.2f} m") if unit == "m" else (lambda v: f"{v:.1f}%")
    rows.append({"Evaluation metric": label,
                 "Published result": f"{fmt(p)}  ({', '.join(fmt(v) for v in p_each)})",
                 "Our result": f"{fmt(o)}  ({', '.join(fmt(v) for v in o_each)})",
                 "Difference": f"{o - p:+.2f} m" if unit == "m" else f"{o - p:+.1f} pp"})
HEADLINE = pd.DataFrame(rows).set_index("Evaluation metric")
display(HEADLINE)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
sizes = ["30x30", "60x60", "100x100"]
for ax, (metric, title) in zip(axes, (("time_s", "Planning time (% vs A*)"),
                                      ("expanded_nodes", "Expanded nodes (% vs A*)"),
                                      ("length_m", "Path length (m vs A*)"),
                                      ("corners_ge45", "Corners >= 45 deg (% vs A*)"))):
    p = [-v for v in mean_reduction(paper_tab, metric)[1]]
    o = ([-v for v in TIMING["time reduction ours (%)"]] if metric == "time_s"
         else [-v for v in mean_reduction(ours_tab, metric)[1]])
    x = np.arange(3)
    ax.bar(x - 0.19, p, 0.36, color=BLUE, label="Published (paper)")
    ax.bar(x + 0.19, o, 0.36, color=ORANGE, label="Reproduced (ours)")
    ax.axhline(0, color=MUTED, lw=0.8); ax.set_xticks(x, sizes); ax.set_title(title)
    ax.grid(axis="y", color=GRID_LINE); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (BLUE, ORANGE)]
fig.legend(handles, ["Published (paper)", "Reproduced (ours)"], frameon=False, ncol=2,
           loc="upper right", bbox_to_anchor=(0.99, 1.08))
fig.suptitle("Adaptive A* relative to A*: published vs reproduced (negative = better)",
             color=INK2, x=0.01, ha="left", y=1.06)
plt.show()

## 9. Robustness on random maps (multiple runs)
The algorithms are deterministic, so the meaningful "multiple runs" vary the **map**. We use the same calibrated generators with **unseen seeds**: 50 maps per scene type, 150 in total. Each method is compared with A\* on the same map using the paired **Wilcoxon signed-rank test**.

In [ ]:
ROB_PAIRS = {"map1": (2, 2, 28, 28), "map2": (2, 2, 58, 58), "map3": (2, 2, 99, 99)}
rob_rows = []
for map_id, (sx, sy, gx, gy) in ROB_PAIRS.items():
    params = {k: tuple(v) if isinstance(v, list) else v for k, v in CALIBRATION[map_id]["params"].items()}
    seed, made = 0, 0
    while made < ROBUST_MAPS:
        seed += 1
        grid = make_map(map_id, 900_000 + seed, **params)
        if grid is None:
            continue
        made += 1
        s, t = from_paper(sx, sy), from_paper(gx, gy)
        for name in ("A*", "Redundant point deletion", "Adaptive A*"):
            spec = PLANNERS[name]
            res = spec.plan(grid, s, t)
            tm = time_planner(spec, grid, s, t, repeats=5)
            rob_rows.append({"map": map_id, "seed": 900_000 + seed, "algorithm": name, "time_s": tm["time_median"],
                             "length_m": res.length, "expanded_nodes": res.expanded,
                             "corners_ge45": res.corners, "raw_cost": res.raw_cost})
ROB = pd.DataFrame(rob_rows)
print(f"{ROB.groupby(['map', 'seed']).ngroups} random maps evaluated ({ROBUST_MAPS} per scene type)")

In [ ]:
summary = []
for map_id in ROB_PAIRS:
    base = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == "A*")].set_index("seed")
    for name in ("Redundant point deletion", "Adaptive A*"):
        other = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == name)].set_index("seed")
        for m in METRICS:
            a, b = base[m].to_numpy(float), other.loc[base.index, m].to_numpy(float)
            change = b - a if m == "length_m" else 100 * (b - a) / a
            p = wilcoxon(a, b).pvalue if np.any(a != b) else 1.0
            summary.append({"map": map_id, "method": "A* + pruning" if name != "Adaptive A*" else name,
                            "metric": m, "mean change": change.mean(), "std": change.std(ddof=1),
                            "better/equal/worse": f"{(b < a).sum()}/{(b == a).sum()}/{(b > a).sum()}",
                            "Wilcoxon p": p})
display(pd.DataFrame(summary).style.format({"mean change": "{:+.2f}", "std": "{:.2f}", "Wilcoxon p": "{:.2g}"}))

# Where do the length gains come from?
a = ROB[ROB["algorithm"] == "A*"].set_index(["map", "seed"])
b = ROB[ROB["algorithm"] == "Adaptive A*"].set_index(["map", "seed"])
c = ROB[ROB["algorithm"] == "Redundant point deletion"].set_index(["map", "seed"])
gap = 100 * (b["raw_cost"] - a["raw_cost"]) / a["raw_cost"]
diff = b["length_m"] - c["length_m"]
display(pd.DataFrame({
    "raw Adaptive A* path suboptimal (maps)": (gap > 1e-9).groupby(level=0).sum(),
    "mean / max suboptimality (%)": gap.groupby(level=0).agg(lambda g: f"{g.mean():.2f} / {g.max():.2f}"),
    "Adaptive A* longer than A*+pruning (maps)": (diff > 1e-9).groupby(level=0).sum(),
    "mean length difference (m)": diff.groupby(level=0).mean().round(2),
}))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, m in zip(axes, METRICS):
    for k, (map_id, size) in enumerate((("map1", "30x30"), ("map2", "60x60"), ("map3", "100x100"))):
        base = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == "A*")].set_index("seed")[m]
        for j, (name, color) in enumerate((("Redundant point deletion", BLUE), ("Adaptive A*", ORANGE))):
            v = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == name)].set_index("seed")[m]
            change = (v - base) if m == "length_m" else 100 * (v - base) / base
            bp = ax.boxplot(change.to_numpy(), positions=[k * 1.2 + j * 0.42], widths=0.34, patch_artist=True,
                            medianprops={"color": INK}, flierprops={"marker": "o", "ms": 2.5,
                                                                    "mfc": color, "mec": color})
            bp["boxes"][0].set(facecolor=color, edgecolor=color, alpha=0.85)
    ax.set_xticks([k * 1.2 + 0.21 for k in range(3)], ["30x30", "60x60", "100x100"])
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title({"time_s": "Time (% vs A*)", "length_m": "Length (m vs A*)",
                  "expanded_nodes": "Expanded nodes (% vs A*)", "corners_ge45": "Corners (% vs A*)"}[m])
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (BLUE, ORANGE)]
fig.legend(handles, ["A* + pruning", "Adaptive A*"], frameon=False, ncol=2,
           loc="upper right", bbox_to_anchor=(0.99, 1.08))
fig.suptitle("Change relative to A* on random maps (negative = better)", color=INK2, x=0.01, ha="left", y=1.06)
plt.show()

## 10. Sensitivity to the ambiguous pruning rule
Section 3.3 of the paper can be read in two ways:
- **anchor** (our default): the anchor node stays until line of sight is lost;
- **single-pass**: the window jumps forward after each deletion.

This choice changes the corner count noticeably.

In [ ]:
rows = []
for size, goal in (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99))):
    s, t = from_paper(2, 2), from_paper(*goal)
    a = PLANNERS["Adaptive A*"].plan(MAPS[size], s, t)
    b = PLANNERS["Adaptive A* (single-pass pruning)"].plan(MAPS[size], s, t)
    paper = PUBLISHED[(PUBLISHED["table"].isin(["T2", "T3", "T4"])) & (PUBLISHED["map"] == size)
                      & (PUBLISHED["algorithm"] == "Adaptive A*")].iloc[0]
    rows.append({"map": size, "corners anchor": a.corners, "corners single-pass": b.corners,
                 "corners paper": int(paper["corners_ge45"]), "length anchor": round(a.length, 2),
                 "length single-pass": round(b.length, 2), "length paper": paper["length_m"]})
display(pd.DataFrame(rows).set_index("map"))

## 11. Summary

| Metric | Verdict |
|---|---|
| Path length | **Comparable** to the paper |
| Expanded nodes | **Same trend**, smaller on 30×30 and 100×100 |
| Planning time | **Lower** reduction than published; Python per-node overhead dominates on small maps |
| Corners | Larger reduction, caused by the pruning interpretation (Section 10), **not** a better method |

Over 150 random maps every claimed effect has the right sign and is statistically significant. Two further findings:
- The raw Adaptive A\* path is **not optimal** on many maps, because the heuristic is inadmissible.
- **A\* + pruning alone** is usually shorter than Adaptive A\*. The length and smoothness gains therefore come mainly from the redundant point deletion, while the adaptive heuristic mainly buys search speed.

The full discussion, with every assumption and deviation, is in the repository: `docs/01_analysis_and_feasibility.md` and `docs/02_reproduction_results.md`.